# 02 — QLoRA Majority Ensembles and Oracle Upper Bound


> **Post-shared-task analysis.** Gold test labels were public when this analysis was
> designed. Nothing in this notebook changes the official CI=0.035 submission or the
> third-place ranking. Results are retrospective and must not be described as untouched
> test-set estimates.

This notebook evaluates fixed majority-vote ensembles of the four submitted QLoRA
adapters. Three-model ensembles use ordinary majority vote. The four-model
ensemble uses the devtest-selected 2,348 adapter for deterministic 2--2 ties.
The oracle is not deployable; it measures complementarity only.

In [ ]:
from pathlib import Path
from collections import Counter
import csv, io, json, os, urllib.request, zipfile

import numpy as np
import pandas as pd


def find_repo_root():
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / 'README.md').exists() and (candidate / 'Test').exists():
            return candidate.resolve()
    raise FileNotFoundError('Run this notebook from the IE2026-HalDetect repository.')


ROOT = find_repo_root()
HERE = ROOT / 'post_task_analysis'
CACHE = HERE / 'cache'
OUTPUT = HERE / 'outputs'
CACHE.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)

GOLD_URL = (
    'https://huggingface.co/datasets/QCRI/ImageEval-ArabicNLP26/'
    'resolve/main/task1b/test_en.jsonl'
)
gold_override = os.getenv('IMAGEEVAL_TEST_GOLD')
GOLD_PATH = Path(gold_override) if gold_override else CACHE / 'test_en.jsonl'
if not GOLD_PATH.exists():
    print('Downloading released gold test JSONL...')
    urllib.request.urlretrieve(GOLD_URL, GOLD_PATH)


def read_gold(path=GOLD_PATH):
    rows = [json.loads(line) for line in Path(path).read_text(encoding='utf-8').splitlines()]
    assert len(rows) == 1000, f'Expected 1,000 gold items, found {len(rows)}'
    frame = pd.DataFrame(rows)
    assert frame['id'].is_unique
    assert frame['labels'].map(lambda x: len(x) == 3 and sum(x) == 1).all()
    frame['gold_idx'] = frame['labels'].map(lambda x: x.index(True))
    return frame


def load_prediction_zip(path, gold):
    path = Path(path)
    assert path.exists(), path
    with zipfile.ZipFile(path) as archive:
        csv_names = [name for name in archive.namelist() if name.lower().endswith('.csv')]
        assert len(csv_names) == 1, (path, csv_names)
        with io.TextIOWrapper(archive.open(csv_names[0]), encoding='utf-8-sig') as handle:
            rows = list(csv.DictReader(handle))
    raw = pd.DataFrame(rows)
    required = {'id', 'statement_index', 'prediction'}
    assert required.issubset(raw.columns), (path, raw.columns)
    raw['statement_index'] = raw['statement_index'].astype(int)
    raw['pred_bool'] = raw['prediction'].str.strip().str.lower().map(
        {'true': True, 'false': False})
    assert raw['pred_bool'].notna().all(), f'Unparseable prediction in {path}'
    assert not raw.duplicated(['id', 'statement_index']).any()
    assert set(raw['statement_index']) == {0, 1, 2}
    grouped = raw.sort_values(['id', 'statement_index']).groupby('id', sort=False)
    vectors = grouped['pred_bool'].apply(list)
    assert vectors.map(len).eq(3).all()
    assert set(vectors.index) == set(gold['id']), f'ID mismatch in {path}'

    result = gold[['id', 'gold_idx']].copy()
    by_id = vectors.to_dict()
    result['pred_vector'] = result['id'].map(by_id)
    result['format_valid'] = result['pred_vector'].map(lambda x: sum(x) == 1)
    result['pred_idx'] = result['pred_vector'].map(
        lambda x: x.index(True) if sum(x) == 1 else np.nan)
    result['correct'] = result['format_valid'] & result['pred_idx'].eq(result['gold_idx'])
    result['error'] = ~result['correct']
    return result


SUBMISSIONS = {
    'Elimination': ROOT / 'Test/qwen2p5-3b-7b/All COT variations/cot-elimination/prediction_en.zip',
    'Socratic': ROOT / 'Test/qwen2p5-3b-7b/All COT variations/cot-socratic/prediction_en.zip',
    'Devils advocate': ROOT / 'Test/qwen2p5-3b-7b/All COT variations/cot-devils-advocate/prediction_en.zip',
    'Evidence first': ROOT / 'Test/qwen2p5-3b-7b/All COT variations/cot-evidence-first/prediction_en.zip',
    'Attribute checklist': ROOT / 'Test/qwen2p5-3b-7b/All COT variations/cot-attribute-checklist/prediction_en.zip',
    'Confidence ranked': ROOT / 'Test/qwen2p5-3b-7b/All COT variations/cot-confidence-ranked/prediction_en.zip',
    'QLoRA 2,000': ROOT / 'Test/qwen2p5-3b-7b/qlora-q7b-2k-image/prediction_en.zip',
    'QLoRA 2,348': ROOT / 'Test/qwen2p5-3b-7b/qlora-q7b-2p3k-image/prediction_en.zip',
    'QLoRA 2,600': ROOT / 'Test/qwen2p5-3b-7b/qlora-q7b-2p6k-image/prediction_en.zip',
    'QLoRA 3,000 legacy': ROOT / 'Test/qwen2p5-3b-7b/qlora-q7b-3k-image/prediction_en.zip',
}

gold = read_gold()
predictions = {name: load_prediction_zip(path, gold) for name, path in SUBMISSIONS.items()}
summary = pd.DataFrame([
    {
        'system': name,
        'n': len(frame),
        'errors': int(frame['error'].sum()),
        'CI': frame['error'].mean(),
        'accuracy': frame['correct'].mean(),
        'format_failures': int((~frame['format_valid']).sum()),
    }
    for name, frame in predictions.items()
]).sort_values(['CI', 'system']).reset_index(drop=True)
display(summary)

In [ ]:
from itertools import combinations

QLORA = ['QLoRA 2,000', 'QLoRA 2,348', 'QLoRA 2,600', 'QLoRA 3,000 legacy']
indexed = {
    name: predictions[name].set_index('id').loc[gold['id']] for name in QLORA
}
pred_matrix = np.stack([indexed[name]['pred_idx'].to_numpy(dtype=int) for name in QLORA])
gold_idx = gold['gold_idx'].to_numpy(dtype=int)


def majority_prediction(member_names, tie_break='QLoRA 2,348'):
    member_rows = [QLORA.index(name) for name in member_names]
    votes = pred_matrix[member_rows]
    tie_row = QLORA.index(tie_break)
    out = []
    for column in votes.T:
        counts = np.bincount(column, minlength=3)
        winners = np.flatnonzero(counts == counts.max())
        out.append(int(winners[0]) if len(winners) == 1 else int(pred_matrix[tie_row, len(out)]))
    return np.asarray(out)


ensemble_rows, ensemble_predictions = [], {}
fixed_memberships = [list(x) for x in combinations(QLORA, 3)] + [QLORA]
for members in fixed_memberships:
    name = ' + '.join(members)
    pred = majority_prediction(members)
    ensemble_predictions[name] = pred
    ensemble_rows.append({
        'ensemble': name,
        'n_members': len(members),
        'tie_breaker': 'QLoRA 2,348' if len(members) % 2 == 0 else 'not needed',
        'errors': int((pred != gold_idx).sum()),
        'CI': (pred != gold_idx).mean(),
        'accuracy': (pred == gold_idx).mean(),
    })

individual_correct = pred_matrix == gold_idx
oracle_correct = individual_correct.any(axis=0)
ensemble_rows.append({
    'ensemble': 'ORACLE: any QLoRA adapter correct',
    'n_members': len(QLORA),
    'tie_breaker': 'not deployable',
    'errors': int((~oracle_correct).sum()),
    'CI': (~oracle_correct).mean(),
    'accuracy': oracle_correct.mean(),
})
ensemble_summary = pd.DataFrame(ensemble_rows).sort_values('CI')
ensemble_summary.to_csv(OUTPUT / 'qlora_ensemble_summary.csv', index=False)
display(ensemble_summary)

output = gold[['id', 'gold_idx']].copy()
for name, pred in ensemble_predictions.items():
    output[name] = pred
output.to_csv(OUTPUT / 'qlora_ensemble_item_predictions.csv', index=False)

## Adapter agreement and complementarity

In [ ]:
agreement = pd.DataFrame(index=QLORA, columns=QLORA, dtype=float)
for a in QLORA:
    for b in QLORA:
        agreement.loc[a, b] = (
            indexed[a]['pred_idx'].to_numpy() == indexed[b]['pred_idx'].to_numpy()
        ).mean()
agreement.to_csv(OUTPUT / 'qlora_prediction_agreement.csv')
display(agreement.round(3))

error_pattern = pd.DataFrame({'id': gold['id']})
for name in QLORA:
    error_pattern[name] = indexed[name]['error'].to_numpy()
error_pattern['n_adapters_wrong'] = error_pattern[QLORA].sum(axis=1)
error_pattern.to_csv(OUTPUT / 'qlora_error_patterns.csv', index=False)
display(error_pattern['n_adapters_wrong'].value_counts().sort_index().rename('items'))